## LoRA Fine-Tune

### Initialization

In [1]:
import random
import numpy as np
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(0)

model_dir = '/openbayes/input/input0'
device = 'cuda'

tokenizer = AutoTokenizer.from_pretrained(model_dir)
tokenizer.pad_token = tokenizer.eos_token

#### See Message Format

In [2]:
messages = [
    {"role": "system", "content": "回答用户的问题。"},
    {"role": "user", "content": '你好呀'},
    {"role": "assistant", "content": "你好，我是 Llama。"},
]
print(tokenizer.apply_chat_template(messages, tokenize=False))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

回答用户的问题。<|eot_id|><|start_header_id|>user<|end_header_id|>

你好呀<|eot_id|><|start_header_id|>assistant<|end_header_id|>

你好，我是 Llama。<|eot_id|><|start_header_id|>assistant<|end_header_id|>




#### Process Dataset

In [3]:
def process_func(system, example, instruction_name='INSTRUCTION', output_name='RESPONSE'):
    MAX_LENGTH = 384
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer(f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{example[instruction_name]}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", add_special_tokens=False)  # add_special_tokens 不在开头加 special_tokens
    response = tokenizer(f"{example[output_name]}<|eot_id|>", add_special_tokens=False)
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]

    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [4]:
import pandas as pd
from datasets import Dataset

df = pd.read_parquet('/openbayes/home/datasets/reasoning_gsm_qna/data.parquet')
ds = Dataset.from_pandas(df)
ds[:1]

{'INSTRUCTION': ['Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'],
 'RESPONSE': ['Natalia sold 48/2 = 24 clips in May.\nNatalia sold 48+24 = 72 clips altogether in April and May.\n\nThe answer will be 72.'],
 'SOURCE': ['gsm8k'],
 'METADATA': ['{"language": "en"}']}

In [5]:
system_prompt = ""

tokenized_id = ds.map(lambda x: process_func(system_prompt, x), remove_columns=ds.column_names)
tokenized_id

Map:   0%|          | 0/8792 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8792
})

In [6]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_id[1]["labels"])))

'Weng earns 12/60 = $0.2 per minute.\nWorking 50 minutes, she earned 0.2 x 50 = $10.\n\nAnswer is 10.<|eot_id|><|eot_id|>'

### Get Model

In [7]:
model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype='auto', device_map=device)
model.enable_input_require_grads()
model

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

#### Set Up LoRA

In [8]:
from peft import LoraConfig, TaskType, get_peft_model  
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"],
    inference_mode=False, 
    r=8,
    lora_alpha=16,
    lora_dropout=0.0
)
config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, lora_alpha=16, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))

In [9]:
model = get_peft_model(model, config)
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features=4096, out_

In [10]:
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


### Train the Model

In [11]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from swanlab_transformers import SwanLabCallback

swanlab_callback = SwanLabCallback(project="llama3-fine-tune")
train_args = TrainingArguments(
    output_dir="/openbayes/home/versions/",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=64,
    weight_decay=0.01,
    logging_steps=1,
    num_train_epochs=1,
    learning_rate=3e-4,
    save_strategy="epoch",
    optim="adamw_torch_fused",
    lr_scheduler_type="cosine",
    warmup_steps=100
)

In [12]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_id,
    callbacks=[swanlab_callback],
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)
trainer.train()

[2024-11-02 17:32:02,915] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/local/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/local/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/usr/local/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/usr/local/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/usr/local/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/usr/local/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/usr/local/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::chrono::_V2::steady_clock::now()@GLIBCXX_3.4.19'
/usr/local/compiler_compat/ld: /usr/local/cuda/lib64/libcu

swanlab: Tracking run with swanlab version 0.3.23                                  
swanlab: Run data will be saved locally in /output/swanlog/run-20241102_173205-a3b1799d
swanlab: 👋 Hi tianzeds, welcome to swanlab!
swanlab: Syncing run horse-8 to the cloud
swanlab: 🌟 Run `swanlab watch /output/swanlog` to view SwanLab Experiment Dashboard locally
swanlab: 🏠 View project at https://swanlab.cn/@tianzeds/llama3-fine-tune
swanlab: 🚀 View run at https://swanlab.cn/@tianzeds/llama3-fine-tune/runs/lfbn4mi7ibq06qdkqnt50


Step,Training Loss
1,1.118900
2,1.109100
3,1.161800
4,1.201500
5,1.135900
6,1.153100
7,1.099900
8,1.135300
9,1.116100
10,1.170400


swanlab: Step 68 on key epoch already exists, ignored.
swanlab: 🌟 Run `swanlab watch /output/swanlog` to view SwanLab Experiment Dashboard locally
swanlab: 🏠 View project at https://swanlab.cn/@tianzeds/llama3-fine-tune
swanlab: 🚀 View run at https://swanlab.cn/@tianzeds/llama3-fine-tune/runs/lfbn4mi7ibq06qdkqnt50


TrainOutput(global_step=68, training_loss=0.7444154422949342, metrics={'train_runtime': 581.7488, 'train_samples_per_second': 15.113, 'train_steps_per_second': 0.117, 'total_flos': 7.805254059452006e+16, 'train_loss': 0.7444154422949342, 'epoch': 0.9899909008189263})

### Inference

In [15]:
prompt = "你是谁？"
messages = [
    {"role": "system", "content": "你是 Llama 3.1。"},
    {"role": "user", "content": prompt}
]
input_ids = tokenizer.apply_chat_template(messages, tokenize=False)
model_inputs = tokenizer([input_ids], return_tensors="pt").to('cuda')
generated_ids = model.generate(model_inputs.input_ids, max_new_tokens=512)
generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


我是 Llama 3.1，Meta 的大型语言模型。
